# CMAFM on M3FD — Kaggle Training Notebook
CFT framework with GPT replaced by CrossModalAttentionFusion (CMAFM).
Dataset: M3FD 6-class, 8:2 split (seed=42), same as original CMAFM paper.

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────
!pip install -q PyYAML>=5.3.1 matplotlib seaborn tqdm tensorboard

In [ ]:
# ── 3. Mount / unzip M3FD dataset ─────────────────────────────
# Upload M3FD_yolo.zip to Kaggle dataset or use input path
import os, zipfile

# Option A: Kaggle Dataset input (recommended)
# Add dataset in the right panel: "Add Data" → your uploaded M3FD_yolo dataset
M3FD_ROOT = '/kaggle/input/m3fd-yolo'   # change to your dataset path

# Option B: upload zip manually
# ZIP_PATH = '/kaggle/working/M3FD_yolo.zip'
# with zipfile.ZipFile(ZIP_PATH) as z:
#     z.extractall('/kaggle/working/M3FD_yolo')
# M3FD_ROOT = '/kaggle/working/M3FD_yolo'

print('M3FD root:', M3FD_ROOT)
!ls {M3FD_ROOT}

In [ ]:
# ── 4. Find & extract CFT_repo.zip ────────────────────────────
import zipfile, os, glob

# CFT_repo.zip 자동 탐색
zip_candidates = glob.glob('/kaggle/input/**/CFT_repo.zip', recursive=True)
print('Found zips:', zip_candidates)

if zip_candidates:
    REPO_ZIP = zip_candidates[0]
    print('Using:', REPO_ZIP)
    with zipfile.ZipFile(REPO_ZIP) as z:
        z.extractall('/kaggle/working/')
else:
    print('ERROR: CFT_repo.zip not found in /kaggle/input/')

# REPO 경로 자동 설정
if os.path.exists('/kaggle/working/CFT_repo'):
    REPO = '/kaggle/working/CFT_repo'
elif os.path.exists('/kaggle/working/train.py'):
    REPO = '/kaggle/working'
else:
    found = glob.glob('/kaggle/working/**/train.py', recursive=True)
    REPO = os.path.dirname(found[0]) if found else '/kaggle/working'

os.chdir(REPO)
print('Working dir:', os.getcwd())
!ls

In [ ]:
# ── 5. Write data yaml + regenerate path txt files ────────────
import yaml, os, shutil
from pathlib import Path

M3FD_ROOT = '/kaggle/input/m3fd-yolo'

# img2label_paths replaces /images/ → /labels/ in the path.
# Image path:  M3FD_ROOT/train/rgb/images/xxxxx.png
# Expected lbl: M3FD_ROOT/train/rgb/labels/xxxxx.txt
# Actual lbl:   M3FD_ROOT/labels/train/rgb/xxxxx.txt
# Fix: copy labels into the expected location under /kaggle/working/m3fd_data/

DATA_DIR = '/kaggle/working/m3fd_data'

for split in ['train', 'val']:
    for mod in ['rgb', 'ir']:
        # symlink images
        img_src = f'{M3FD_ROOT}/{split}/{mod}/images'
        img_dst = f'{DATA_DIR}/{split}/{mod}/images'
        os.makedirs(img_dst, exist_ok=True)
        for f in Path(img_src).glob('*'):
            dst_f = Path(img_dst) / f.name
            if not dst_f.exists():
                os.symlink(f, dst_f)

        # copy labels into /images/../labels/ structure
        lbl_src = f'{M3FD_ROOT}/labels/{split}/{mod}'
        lbl_dst = f'{DATA_DIR}/{split}/{mod}/labels'
        os.makedirs(lbl_dst, exist_ok=True)
        for f in Path(lbl_src).glob('*.txt'):
            dst_f = Path(lbl_dst) / f.name
            if not dst_f.exists():
                shutil.copy2(f, dst_f)

print('Data structure ready.')

# Regenerate txt files
def write_txt(img_dir, out_path):
    paths = sorted(Path(img_dir).glob('*.png')) + sorted(Path(img_dir).glob('*.jpg'))
    with open(out_path, 'w') as f:
        f.write('\n'.join(str(p) for p in paths))
    print(f'  {out_path}: {len(paths)} images')

txt_dir = '/kaggle/working/m3fd_txt'
os.makedirs(txt_dir, exist_ok=True)
write_txt(f'{DATA_DIR}/train/rgb/images', f'{txt_dir}/train_rgb.txt')
write_txt(f'{DATA_DIR}/val/rgb/images',   f'{txt_dir}/val_rgb.txt')
write_txt(f'{DATA_DIR}/train/ir/images',  f'{txt_dir}/train_ir.txt')
write_txt(f'{DATA_DIR}/val/ir/images',    f'{txt_dir}/val_ir.txt')

data_cfg = {
    'train_rgb': f'{txt_dir}/train_rgb.txt',
    'val_rgb':   f'{txt_dir}/val_rgb.txt',
    'train_ir':  f'{txt_dir}/train_ir.txt',
    'val_ir':    f'{txt_dir}/val_ir.txt',
    'nc': 6,
    'names': ['People', 'Car', 'Bus', 'Motorcycle', 'Lamp', 'Truck']
}

os.makedirs('data/multispectral', exist_ok=True)
with open('data/multispectral/M3FD_kaggle.yaml', 'w') as f:
    yaml.dump(data_cfg, f)
print('Data yaml written.')

In [ ]:
# ── 6. Verify model forward pass ──────────────────────────────
import sys
sys.path.insert(0, REPO)

import torch
from models.yolo import Model

m = Model('models/transformer/yolov5l_cmafm_M3FD.yaml', ch=3, nc=6)
m.eval()
x = torch.zeros(1, 3, 640, 640)
with torch.no_grad():
    out = m(x)
print('Model OK, params:', sum(p.numel() for p in m.parameters())/1e6, 'M')
print('Output shape:', out[0].shape)  # (1, 25200, 11)
del m

In [ ]:
# ── 7. Train ──────────────────────────────────────────────────
import os, yaml, glob

# REPO 재설정 (세션 끊긴 후 재실행 시 대비)
if 'REPO' not in dir():
    if os.path.exists('/kaggle/working/CFT_repo'):
        REPO = '/kaggle/working/CFT_repo'
    else:
        found = glob.glob('/kaggle/working/**/train.py', recursive=True)
        REPO = os.path.dirname(found[0]) if found else '/kaggle/working'
    os.chdir(REPO)
    print('REPO set to:', REPO)

# hyp.scratch.yaml: 없으면 직접 생성
hyp_path = f'{REPO}/data/hyp.scratch.yaml'
if not os.path.exists(hyp_path):
    os.makedirs(f'{REPO}/data', exist_ok=True)
    with open(hyp_path, 'w') as f:
        f.write("lr0: 0.01\nlrf: 0.2\nmomentum: 0.937\nweight_decay: 0.0005\n"
                "warmup_epochs: 3.0\nwarmup_momentum: 0.8\nwarmup_bias_lr: 0.1\n"
                "box: 0.05\ncls: 0.5\ncls_pw: 1.0\nobj: 1.0\nobj_pw: 1.0\n"
                "iou_t: 0.20\nanchor_t: 4.0\nfl_gamma: 0.0\n"
                "hsv_h: 0.015\nhsv_s: 0.7\nhsv_v: 0.4\n"
                "degrees: 0.0\ntranslate: 0.1\nscale: 0.5\nshear: 0.0\n"
                "perspective: 0.0\nflipud: 0.0\nfliplr: 0.5\nmosaic: 1.0\nmixup: 0.0\n")
    print('Created hyp.scratch.yaml')

# M3FD_kaggle.yaml: 없으면 재생성
data_yaml = f'{REPO}/data/multispectral/M3FD_kaggle.yaml'
os.makedirs(f'{REPO}/data/multispectral', exist_ok=True)
if not os.path.exists(data_yaml):
    txt_dir = '/kaggle/working/m3fd_txt'
    data_cfg = {
        'train_rgb': f'{txt_dir}/train_rgb.txt',
        'val_rgb':   f'{txt_dir}/val_rgb.txt',
        'train_ir':  f'{txt_dir}/train_ir.txt',
        'val_ir':    f'{txt_dir}/val_ir.txt',
        'nc': 6,
        'names': ['People', 'Car', 'Bus', 'Motorcycle', 'Lamp', 'Truck']
    }
    with open(data_yaml, 'w') as f:
        yaml.dump(data_cfg, f)
    print('Recreated M3FD_kaggle.yaml')

# resume 여부 판단
last_pt = '/kaggle/working/runs/train/cmafm_m3fd/weights/last.pt'
if os.path.exists(last_pt):
    print(f'Resuming from {last_pt}')
    resume_arg = f'--resume {last_pt}'
    weights_arg = ''
    cfg_arg = ''
    data_arg = ''
    hyp_arg = ''
else:
    print('Starting fresh training')
    resume_arg = ''
    weights_arg = "--weights ''"
    cfg_arg = f'--cfg models/transformer/yolov5l_cmafm_M3FD.yaml'
    data_arg = f'--data {data_yaml}'
    hyp_arg = f'--hyp {hyp_path}'

os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

!python train.py \
    {resume_arg} \
    {cfg_arg} \
    {data_arg} \
    {hyp_arg} \
    {weights_arg} \
    --epochs 30 \
    --batch-size 16 \
    --img-size 640 \
    --project /kaggle/working/runs/train \
    --name cmafm_m3fd \
    --exist-ok \
    --workers 4 \
    2>&1 | tee /kaggle/working/train_log.txt

In [ ]:
# ── 8. Show results ───────────────────────────────────────────
import pandas as pd

results_csv = '/kaggle/working/runs/train/cmafm_m3fd/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

# mAP@0.5 per epoch
print(df[['epoch', 'metrics/mAP_0.5', 'metrics/mAP_0.5:0.95']].tail(10).to_string())
print('\nBest mAP@0.5:', df['metrics/mAP_0.5'].max())

In [ ]:
# ── 9. Plot training curve ────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['epoch'], df['train/box_loss'], label='box')
axes[0].plot(df['epoch'], df['train/obj_loss'], label='obj')
axes[0].plot(df['epoch'], df['train/cls_loss'], label='cls')
axes[0].set_title('Train Loss'); axes[0].legend()

axes[1].plot(df['epoch'], df['metrics/mAP_0.5'], label='mAP@0.5', color='green')
axes[1].set_title('Validation mAP@0.5'); axes[1].legend()
plt.tight_layout()
plt.savefig('/kaggle/working/training_curve.png', dpi=150)
plt.show()
print('Saved training_curve.png')

In [ ]:
# ── 10. Save outputs ─────────────────────────────────────────
import shutil
OUT_DIR = '/kaggle/working/runs/train/cmafm_m3fd'
print('Best weights:', OUT_DIR + '/weights/best.pt')
print('Last weights:', OUT_DIR + '/weights/last.pt')
# These are automatically saved as Kaggle output files